## Data acquisition

Pulls weekly initial unemployment insurance claims for California, Texas, 
and Florida from FRED, and unemployment-related Google Trends data via 
pytrends. Trends data is cached to CSV so the analysis is reproducible 
without re-hitting the API.

In [5]:
import sys
!{sys.executable} -m pip install fredapi pandas pytrends scikit-learn lightgbm statsmodels matplotlib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 28.5 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.1/8.1 MB 56.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 43.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 35.8 MB/s  0:00:00eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.2/5.2 MB 27.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.2/8.2 MB 52.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 51.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 46.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.3/20.3 MB 31.2 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.6/8.6 MB 45.5 MB/s  0:00:00
   ━━━━━━╸━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  4/24 [numpy]]  WARNING: The scripts f2py and numpy-config are installed in '/Library/Frameworks/Python.framework/Versions/3.14/bin' which is not on PATH.
  Consider adding this

In [ ]:
from fredapi import Fred
import pandas as pd

fred = Fred(api_key='YOUR_KEY_HERE')

ca_claims = fred.get_series('CAICLAIMS')
tx_claims = fred.get_series('TXICLAIMS')
fl_claims = fred.get_series('FLICLAIMS')

claims = pd.DataFrame({
    'CA': ca_claims,
    'TX': tx_claims,
    'FL': fl_claims
})

claims.tail()

,CA,TX,FL
2026-03-21,37819.0,15536.0,5398.0
2026-03-28,38010.0,17488.0,5110.0
2026-04-04,40140.0,16189.0,5584.0
2026-04-11,40874.0,17263.0,6387.0
2026-04-18,42464.0,16233.0,5659.0


## Google Trends pull (cached)

Queries pytrends for unemployment-related search terms in California and 
caches the result to CSV. The five-terms-per-query cap means the six-term 
set requires either two queries with an anchor term for normalization or 
a reduction to five terms; this initial pull uses five.

In [4]:
from pytrends.request import TrendReq

pytrends = TrendReq(hl='en-US', tz=360)
terms = ['unemployment benefits', 'file for unemployment', 'how to file unemployment', 'laid off', 'severance']
pytrends.build_payload(terms, timeframe='2018-01-01 2026-04-01', geo='US-CA')
ca_trends = pytrends.interest_over_time()
ca_trends.tail()

,unemployment benefits,file for unemployment,how to file unemployment,laid off,severance,isPartial
date,,,,,,
2025-12-01,5,1,1,3,11,False
2026-01-01,5,2,1,3,12,False
2026-02-01,5,1,1,4,11,False
2026-03-01,5,1,0,4,10,False
2026-04-01,4,1,0,4,10,True
